In [0]:
%run ./00_config

In [0]:
from pyspark.sql import functions as F

df_gold = spark.read.format("delta").load(GOLD_PATH)

latest_run_path = latest_bronze_run_path()
latest_run_id = latest_run_path.split("/")[-1]  # extract just the run_id folder name from the full path

quarantine_run_folder = f"{QUARANTINE_PATH}/{latest_run_id}"
df_quarantine = spark.read.format("delta").load(quarantine_run_folder)

print(f"Testing against run: {latest_run_id}")
print(f"Gold rows: {df_gold.count()}")
print(f"Quarantine rows: {df_quarantine.count()}")

In [0]:
quarantined_ids = [row["chapter_id"] for row in df_quarantine.select("chapter_id").collect()]
gold_ids = [row["chapter_id"] for row in df_gold.select("chapter_id").collect()]

overlap = set(quarantined_ids) & set(gold_ids)

assert len(overlap) == 0, f"FAIL: These quarantined chapter_ids leaked into Gold: {overlap}"

print(f"PASS: No quarantined chapter_ids found in Gold. Quarantined: {quarantined_ids}")

In [0]:
# NOTE: This test assumes the pipeline was run with synthetic test rows injected
# (see 02_bronze_to_silver.py). It verifies DQ-W1 LOGIC works correctly, not that
# warnings must always exist in production — a real run with fully clean data
# would legitimately have zero WARNING rows, and that's NOT a failure.


warning_rows = df_gold.filter(F.col("dq_status") == DQ_STATUS_WARNING).collect()

assert len(warning_rows) > 0, "FAIL: No WARNING rows found in Gold — expected at least one from DQ-W1 test case."

for row in warning_rows:
    assert row["dq_warnings"] == DQ_REASON_MISSING_UNKNOWN_CITY, \
        f"FAIL: Warning row {row['chapter_id']} has unexpected dq_warnings value: {row['dq_warnings']}"

print(f"PASS: Found {len(warning_rows)} WARNING row(s) in Gold, correctly tagged: {[r['chapter_id'] for r in warning_rows]}")

In [0]:
actual_gold_columns = set(df_gold.columns)
expected_gold_columns = set(GOLD_COLUMNS)

assert actual_gold_columns == expected_gold_columns, (
    f"FAIL: Gold schema mismatch.\n"
    f"Missing columns: {expected_gold_columns - actual_gold_columns}\n"
    f"Unexpected extra columns: {actual_gold_columns - expected_gold_columns}"
)

print(f"PASS: Gold schema exactly matches the contract: {sorted(actual_gold_columns)}")